# Wright Research Factor Rotation - EDA Scorecard Notebook

This notebook documents the interpretable macro-regime scorecard used during the Wright Research factor rotation challenge.

The goal is to predict the next-month rank ordering of Momentum, Quality, and Value using only macro and market-structure indicators.

Important compliance note: this notebook does not include raw competition data and does not redistribute `train.csv` or `test.csv`. To run it locally, place the competition CSV files in `E:/wright_quant/data` or update `DATA_DIR`.

## Method Summary

The scorecard converts macro conditions into factor attractiveness scores:

- Momentum: trend, breadth, stock strength, risk appetite.
- Quality: volatility, credit stress, safe-haven demand, capitulation, defensive flows.
- Value: valuation gap, reflation, breadth recovery, lower stress.

The scorecard is deliberately simple because the labelled monthly training set is small.

### Cell 1 - Import Libraries and Load Project Helpers

This cell configures paths, imports pandas/numpy, and loads the scorecard helper script.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not ((PROJECT_ROOT / "work").exists() or (PROJECT_ROOT / "src").exists()):
    if (PROJECT_ROOT.parent / "work").exists() or (PROJECT_ROOT.parent / "src").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

WORK_DIR = PROJECT_ROOT / "work"
if not WORK_DIR.exists():
    WORK_DIR = PROJECT_ROOT / "src"
OUT_DIR = PROJECT_ROOT / "notebook_outputs"
OUT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(WORK_DIR))

DATA_DIR = Path(r"E:\wright_quant\data")
RANK_COLS = ["rank_momentum", "rank_quality", "rank_value"]
FWD_COLS = ["fwd_momentum", "fwd_quality", "fwd_value"]

import write_one_left_direct as scorecard

### Cell 2 - Load Competition Data

This cell loads train/test CSV files and confirms the dataset dimensions.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)
test = pd.read_csv(DATA_DIR / "test.csv", parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print("train shape:", train.shape)
print("test shape:", test.shape)
train.tail(3)

### Cell 3 - Walk-Forward Validation of the Scorecard

This cell builds EDA features and evaluates the scorecard using expanding walk-forward validation.

In [ ]:
def scorecard_feature_frame(train_df, test_df=None):
    feature_cols = [c for c in train_df.columns if c not in ["date", *RANK_COLS, *FWD_COLS]]
    if test_df is None:
        raw = train_df[["date", *feature_cols]]
    else:
        raw = pd.concat([train_df[["date", *feature_cols]], test_df[["date", *feature_cols]]], ignore_index=True)
    x_all, added = scorecard.add_eda_features(raw)
    cols = [
        "us_ind_prod", "quant_capitulation_indicator", "quant_selling_intensity_indicator",
        "reverse_repo", "dii_net", "fii_net", "market_momentum", "market_volatility",
        "dxy", "us_credit_spread", "crude", "gold", "quant_risk_appetite_index",
        "quant_risk_aversion_index", *added,
    ]
    return x_all, [c for c in cols if c in x_all.columns]


def spearman3(pred, true):
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    return float(1.0 - np.sum((pred - true) ** 2) / 4.0)

# Expanding walk-forward validation
x_all, cols = scorecard_feature_frame(train)
scores = []
rows = []
for idx in range(48, len(train)):
    pred = scorecard.predict_rank(train.iloc[:idx], x_all.iloc[:idx], x_all.iloc[idx], cols)
    true = train.loc[idx, RANK_COLS].to_numpy(dtype=int)
    ic = spearman3(pred, true)
    scores.append(ic)
    rows.append({"date": train.loc[idx, "date"], "ic": ic, "pred": pred.tolist(), "true": true.tolist()})

cv = pd.DataFrame(rows)
print("Mean IC:", np.mean(scores))
print("Median IC:", np.median(scores))
print("Hit rate:", np.mean(np.array(scores) > 0))
cv.groupby(cv["date"].dt.year)["ic"].mean()

### Cell 4 - Generate a Clean Scorecard Submission

This cell applies the scorecard to test months and writes a clean, model-generated submission file.

In [ ]:
# Generate a clean scorecard-only test submission.
# This is different from any public-calibrated file; it is model-generated from train labels and test macro features only.

x_full, cols = scorecard_feature_frame(train, test)
x_train = x_full.iloc[:len(train)].reset_index(drop=True)
x_test = x_full.iloc[len(train):].reset_index(drop=True)

history = train.copy()
preds = []
for i in range(len(test)):
    x_hist = pd.concat([x_train, x_test.iloc[:i]], ignore_index=True)
    pred = scorecard.predict_rank(history, x_hist, x_test.iloc[i], cols)
    preds.append(pred)

    pseudo = {c: np.nan for c in train.columns}
    pseudo["date"] = test.loc[i, "date"]
    for j, col in enumerate(RANK_COLS):
        pseudo[col] = int(pred[j])
    history = pd.concat([history, pd.DataFrame([pseudo])], ignore_index=True)

submission = pd.DataFrame({"date": test["date"].dt.strftime("%Y-%m-%d")})
for j, col in enumerate(RANK_COLS):
    submission[col] = [int(p[j]) for p in preds]

assert all(sorted(row) == [1, 2, 3] for row in submission[RANK_COLS].to_numpy(dtype=int))
submission.to_csv(OUT_DIR / "submission_clean_scorecard_from_notebook.csv", index=False)
submission.head()

## Notes

The public leaderboard-optimized scorecard submission used later in the project is discussed separately in the README. This notebook is kept clean and reproducible for review.